# Stage 5 — Leakage-Safe Feature Pool Construction

This stage constructs a large cross-sectional pool of lagged predictor values. It does **not** use the test-set target to choose predictors.

Key choices:

- Use exact predictor levels from **2022** only.
- Use exact recent changes from **2018 to 2022** only.
- Remove indicators with more than **50% missingness** in the 2022 predictor value.
- Remove engineered predictor columns with more than **50% missingness** in the training data.
- Remove target-like indicators such as mortality, death, survival, life expectancy, probability of dying, and close child mortality outcomes.

The supervised step that ranks indicators by correlation with the target is delayed until after the train/test split, and it is performed on training countries only.

## 5.1 Configuration for feature construction and selection

In [668]:
MAX_INDICATOR_MISSINGNESS = 0.50
MAX_FINAL_FEATURE_MISSINGNESS = 0.50
CANDIDATE_LIMIT = 60
FINAL_INDICATOR_LIMIT = 25
FINAL_ENGINEERED_FEATURE_LIMIT = 25
REDUNDANCY_LIMIT = 0.90
ENGINEERED_REDUNDANCY_LIMIT = 0.90

LEVEL_VALUE_COLUMN = f"value_{PREDICTOR_LEVEL_YEAR}"
CHANGE_VALUE_COLUMN = f"change_{CHANGE_START_YEAR}_{CHANGE_END_YEAR}"

print("Target year:", TARGET_YEAR)
print("Predictor level year:", PREDICTOR_LEVEL_YEAR)
print("Change feature:", CHANGE_START_YEAR, "to", CHANGE_END_YEAR)
print("Maximum allowed indicator missingness:", MAX_INDICATOR_MISSINGNESS)
print("Maximum allowed final engineered-feature missingness:", MAX_FINAL_FEATURE_MISSINGNESS)
print("Candidate indicator shortlist size:", CANDIDATE_LIMIT)
print("Maximum final original WDI indicators:", FINAL_INDICATOR_LIMIT)
print("Maximum final engineered predictor columns:", FINAL_ENGINEERED_FEATURE_LIMIT)
print("Maximum allowed redundancy among engineered predictors:", ENGINEERED_REDUNDANCY_LIMIT)

# Exact 2022 values are used for final feature screening and modeling.
all_2022_values = fixed_year_values(country_wdi, PREDICTOR_LEVEL_YEAR, value_name=LEVEL_VALUE_COLUMN)
all_2022_values = all_2022_values[all_2022_values["country_code"].isin(target_df["country_code"])].copy()

target_country_count = target_df["country_code"].nunique()
indicator_missingness_2022 = (
    all_2022_values
    .groupby(["indicator_code", "indicator_name"], as_index=False)
    .agg(observed_countries=(LEVEL_VALUE_COLUMN, "count"))
)
indicator_missingness_2022["missing_countries"] = target_country_count - indicator_missingness_2022["observed_countries"]
indicator_missingness_2022["missing_pct"] = indicator_missingness_2022["missing_countries"] / target_country_count
indicator_missingness_2022["source_year"] = PREDICTOR_LEVEL_YEAR
indicator_missingness_2022 = indicator_missingness_2022.sort_values("missing_pct")

indicator_missingness_2022.to_csv(TABLE_DIR / "indicator_missingness_2022.csv", index=False)
display(indicator_missingness_2022.head(10))

Target year: 2023
Predictor level year: 2022
Change feature: 2018 to 2022
Maximum allowed indicator missingness: 0.5
Maximum allowed final engineered-feature missingness: 0.5
Candidate indicator shortlist size: 60
Maximum final original WDI indicators: 25
Maximum final engineered predictor columns: 25
Maximum allowed redundancy among engineered predictors: 0.9


,indicator_code,indicator_name,observed_countries,missing_countries,missing_pct,source_year
1142,WB_WDI_SP_POP_80UP_MA_5Y,"Population ages 80 and above, male (% of male ...",196,0,0.0,2022
1095,WB_WDI_SP_POP_0014_MA_IN,"Population ages 0-14, male",196,0,0.0,2022
1096,WB_WDI_SP_POP_0014_MA_ZS,"Population ages 0-14, male (% of male population)",196,0,0.0,2022
1097,WB_WDI_SP_POP_0014_TO,"Population ages 0-14, total",196,0,0.0,2022
1098,WB_WDI_SP_POP_0014_TO_ZS,Population ages 0-14 (% of total population),196,0,0.0,2022
1099,WB_WDI_SP_POP_0509_FE_5Y,"Population ages 05-09, female (% of female pop...",196,0,0.0,2022
1100,WB_WDI_SP_POP_0509_MA_5Y,"Population ages 05-09, male (% of male populat...",196,0,0.0,2022
1101,WB_WDI_SP_POP_1014_FE_5Y,"Population ages 10-14, female (% of female pop...",196,0,0.0,2022
1102,WB_WDI_SP_POP_1014_MA_5Y,"Population ages 10-14, male (% of male populat...",196,0,0.0,2022
1103,WB_WDI_SP_POP_1519_FE_5Y,"Population ages 15-19, female (% of female pop...",196,0,0.0,2022


## 5.2 Remove target-like and leakage-risk indicators

In [670]:
def is_target_like_indicator(indicator_name, indicator_code):
    """Flag indicators that are too close to the U5MR target or child-mortality outcomes."""
    text = f"{indicator_name} {indicator_code}".lower()
    leakage_terms = [
        "mortality", "death", "deaths", "dying", "survival", "life expectancy", "lifespan",
        "infant mortality", "neonatal", "under-5", "under five", "under-five",
        "stillbirth", "hci", "human capital index"
    ]
    return any(term in text for term in leakage_terms)

eligible_indicator_base = indicator_missingness_2022.copy()
eligible_indicator_base["target_like"] = eligible_indicator_base.apply(
    lambda row: is_target_like_indicator(row["indicator_name"], row["indicator_code"]),
    axis=1
)

eligible_indicator_base["passes_missingness"] = eligible_indicator_base["missing_pct"] <= MAX_INDICATOR_MISSINGNESS

eligible_indicators = eligible_indicator_base[
    eligible_indicator_base["passes_missingness"] & (~eligible_indicator_base["target_like"])
].copy()

excluded_target_like = eligible_indicator_base[eligible_indicator_base["target_like"]].copy()
excluded_high_missing = eligible_indicator_base[~eligible_indicator_base["passes_missingness"]].copy()

eligible_indicators.to_csv(TABLE_DIR / "eligible_indicators_after_2022_missingness_and_leakage_filters.csv", index=False)
excluded_target_like.to_csv(TABLE_DIR / "excluded_target_like_indicators.csv", index=False)
excluded_high_missing.to_csv(TABLE_DIR / "excluded_high_2022_missingness_indicators.csv", index=False)

print("Indicators with 2022 observations before filters:", len(indicator_missingness_2022))
print("Indicators after <=50% 2022 missingness and leakage filters:", len(eligible_indicators))
print("Target-like indicators excluded:", len(excluded_target_like))
print("High-2022-missingness indicators excluded:", len(excluded_high_missing))

Indicators with 2022 observations before filters: 1251
Indicators after <=50% 2022 missingness and leakage filters: 874
Target-like indicators excluded: 32
High-2022-missingness indicators excluded: 347


## 5.3 Build 2022-level and 2018-to-2022-change feature pools

In [672]:
eligible_codes = set(eligible_indicators["indicator_code"])

level_2022_pool = all_2022_values[all_2022_values["indicator_code"].isin(eligible_codes)].copy()

change_2018_2022_pool = exact_change_between_years(country_wdi, CHANGE_START_YEAR, CHANGE_END_YEAR)
change_2018_2022_pool = change_2018_2022_pool[
    change_2018_2022_pool["country_code"].isin(target_df["country_code"])
    & change_2018_2022_pool["indicator_code"].isin(eligible_codes)
].copy()

level_2022_pool.to_csv(DATA_DIR / "feature_pool_2022_level_long.csv", index=False)
change_2018_2022_pool.to_csv(DATA_DIR / "feature_pool_change_2018_2022_long.csv", index=False)

print("2022-level feature pool:", level_2022_pool.shape)
print("2018-to-2022 change feature pool:", change_2018_2022_pool.shape)

2022-level feature pool: (140971, 6)
2018-to-2022 change feature pool: (136515, 9)
